In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import csv
import re
from tqdm import tqdm
import json

In [2]:
headers = {
'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36'
}

#defining requests session
session = requests.Session()
session.headers = headers

In [3]:
urls = []
with open('urls.csv', newline='') as f:
    reader = csv.reader(f)    
    for entry in reader:
        urls.append(entry[0])

In [4]:
len(urls)

79680

In [5]:
test_urls = urls[:10]

In [6]:
test_urls

['https://www.immoweb.be/en/classified/new-real-estate-project-apartments/for-sale/bruxelles/1000/20529724',
 'https://www.immoweb.be/en/classified/new-real-estate-project-apartments/for-sale/bruxelles/1000/20632916',
 'https://www.immoweb.be/en/classified/new-real-estate-project-apartments/for-sale/bruxelles/1000/20633255',
 'https://www.immoweb.be/en/classified/new-real-estate-project-apartments/for-sale/bruxelles/1000/20632507',
 'https://www.immoweb.be/en/classified/new-real-estate-project-houses/for-sale/bruxelles/1000/20633288',
 'https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20634289',
 'https://www.immoweb.be/en/classified/duplex/for-sale/bruxelles/1000/20634186',
 'https://www.immoweb.be/en/classified/duplex/for-sale/bruxelles/1000/20634197',
 'https://www.immoweb.be/en/classified/duplex/for-sale/bruxelles/1000/20634176',
 'https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20634170']

In [5]:
response = session.get(test_urls[-1])
soup = bs(response.text, "html.parser")

NameError: name 'test_urls' is not defined

In [8]:
test_urls[5:]

['https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20634289',
 'https://www.immoweb.be/en/classified/duplex/for-sale/bruxelles/1000/20634186',
 'https://www.immoweb.be/en/classified/duplex/for-sale/bruxelles/1000/20634197',
 'https://www.immoweb.be/en/classified/duplex/for-sale/bruxelles/1000/20634176',
 'https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20634170']

In [43]:
res = session.get('https://www.immoweb.be/en/classified/villa/for-sale/langemark-poelkapelle/8920/20638382')
sup = bs(res.text, "html.parser")

info_script = sup.find('script', string=re.compile("av_items = \[\{"))
express = '"wellnessEquipment": \{\n.*"hasSwimmingPool":.*"'
pool = re.findall(express, info_script.string)[0][-5:-1]

                    
print(pool == 'true')
print(sup.prettify())

True
<!DOCTYPE html>
<html itemscope="" itemtype="http://schema.org/WebPage" lang="en">
 <head>
  <meta charset="utf-8"/>
  <script>
   if (!Array.prototype.flat) { window.location.replace('https://www.immoweb.be/en/outdated-browser') }
  </script>
  <script type="text/javascript">
   const variantName = "";
    const isABTestWithCookie = false;

    if (!variantName || isABTestWithCookie) { // AB test is not configured or started using cookie for storing variant
        localStorage.removeItem("ab-test");
        window.ABTestVariant = null;
    } else {
        setABTestVariant();
    }

    function setABTestVariant() {
        const abTestInDedicatedRoute = [];
        const currentRoute = "classified details";
        if (abTestInDedicatedRoute && !abTestInDedicatedRoute.includes(currentRoute)) {
            // AB test is not configured for current route
            return;
        }
        
        const setVariantInLocalStorage = () => {
            const variant = null || 0;



In [11]:
def scrappy(html_text):    
    soup = bs(html_text, "html.parser")
    print(soup.prettify())
    info_script = soup.find('script', string=re.compile("av_items = \[\{"))
    expr = re.compile("av_items = \[\{[\S\n ]+\}\]")
    dict_like = re.findall(expr, info_script.string)[0]
    dict_like = re.sub('av_items = \[', '', dict_like)
    dict_like = re.sub('\]', '', dict_like)
    dict_like = re.sub('"list_name":.*,\n', '', dict_like)
    dict_like = re.sub(',\n.*\}', '}', dict_like)    
    prop_dict = (json.loads(dict_like))

    url_list = []
    real_estate = {'immo_id': None, 'zip_code': None, 'province':None, 'price': None, 'subtype_of_property': None, 'building_condition': None, 'living_area': None, 'year_of_construction': None, 
               'energy_certificate': None, 'geolocation': None, 'equipped_kitchen': None, 'bedroom_nr': None, 'swimming_pool': 0, 'terrace': 0, 'garden': 0, 'plot_surface': 0}

    annuitant_check = soup.find_all('th', string=re.compile("annuitant"))
    title = soup.find('title').text    

    if len(annuitant_check) > 0:
        return False
    
    elif len(re.findall("(\s\d+m)", title)) > 0:
        real_estate['immo_id'] = int(prop_dict['id'])
        real_estate['zip_code'] = int(prop_dict['zip_code'])
        real_estate['province'] = prop_dict['province']
        real_estate['price'] = int(prop_dict['price'])
        real_estate['subtype_of_property'] = prop_dict['subtype']
        real_estate['building_condition'] = prop_dict['building_state']
        real_estate['living_area'] = int(prop_dict['indoor_surface'])
        if prop_dict['year_of_construction'] != '':
            real_estate['year_of_construction'] = int(prop_dict['year_of_construction'])
        real_estate['energy_certificate'] = prop_dict['energy_certificate']
        real_estate['geolocation'] = prop_dict['geolocation']
        real_estate['equipped_kitchen'] = prop_dict['kitchen_type']
        real_estate['bedroom_nr'] = int(prop_dict['nb_bedrooms'])
        pool_indicator = re.findall('"wellnessEquipment": \{\n.*"hasSwimmingPool":.*"', info_script.string)[0][-5:-1]
        if pool_indicator == 'true':
            real_estate['swimming_pool'] = 1
        if prop_dict['outdoor_terrace_exists'] == 'true':
            real_estate['terrace'] = 1
        if prop_dict['outdoor_surface'] != '':
            real_estate['garden'] = int(prop_dict['outdoor_surface'])
        if prop_dict['land_surface'] != '':
            real_estate['plot_surface'] = int(prop_dict['land_surface'])

        return real_estate

    elif 'group' in prop_dict['subtype']:
        anchor = soup.find('template', string=re.compile("All properties"))
        for link in anchor.parent.find_all('a'):
            url_list.append([link['href']])
        
        return url_list
    

In [9]:
response = session.get('https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20637128')
#soup = bs(response.text, "html.parser")

In [12]:
scrappy(response.text)

<!DOCTYPE html>
<html itemscope="" itemtype="http://schema.org/WebPage" lang="en">
 <head>
  <meta charset="utf-8"/>
  <script>
   if (!Array.prototype.flat) { window.location.replace('https://www.immoweb.be/en/outdated-browser') }
  </script>
  <script type="text/javascript">
   const variantName = "";
    const isABTestWithCookie = false;

    if (!variantName || isABTestWithCookie) { // AB test is not configured or started using cookie for storing variant
        localStorage.removeItem("ab-test");
        window.ABTestVariant = null;
    } else {
        setABTestVariant();
    }

    function setABTestVariant() {
        const abTestInDedicatedRoute = [];
        const currentRoute = "classified details";
        if (abTestInDedicatedRoute && !abTestInDedicatedRoute.includes(currentRoute)) {
            // AB test is not configured for current route
            return;
        }
        
        const setVariantInLocalStorage = () => {
            const variant = null || 0;

     

{'immo_id': 20637128,
 'zip_code': 1000,
 'province': 'Brussels',
 'price': 275000,
 'subtype_of_property': 'apartment',
 'building_condition': 'AS_NEW',
 'living_area': 57,
 'year_of_construction': None,
 'energy_certificate': 'E',
 'geolocation': '4.346507,50.846607',
 'equipped_kitchen': 'installed',
 'bedroom_nr': 1,
 'swimming_pool': 0,
 'terrace': 0,
 'garden': 0,
 'plot_surface': 0}

In [52]:
def main_scrapper():

    headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36'
    }

    #defining requests session
    session = requests.Session()
    session.headers = headers

    field_names = ['immo_id', 'zip_code', 'province', 'price', 'subtype_of_property', 'building_condition', 'living_area', 'year_of_construction', 'energy_certificate', 'geolocation', 
                   'equipped_kitchen', 'bedroom_nr', 'swimming_pool', 'terrace', 'garden', 'plot_surface']

    with open('property_data.csv', 'a', newline='') as file:
        csv.writer(file).writerow(field_names)

    urls = []
    with open('urls.csv', newline='') as f:
        reader = csv.reader(f)    
        for entry in reader:
            urls.append(entry[0])

    for url in tqdm(urls[:20]):
        response = session.get(url)
        try:
            result = scrappy(response.text)
            if isinstance(result, dict):
                with open('property_data.csv', 'a', newline='') as file:
                    csv.DictWriter(file, fieldnames=field_names).writerow(result)
            elif isinstance(result, list):
                with open('additional_urls.csv', 'a', newline='') as file:
                    for link in result:
                        csv.writer(file).writerow(link)
        except Exception as ex:
            with open('failed_urls.csv', 'a', newline='') as file:
                csv.writer(file).writerow([url])        
            continue

In [51]:
field_names = ['immo_id', 'zip_code', 'province', 'price', 'subtype_of_property', 'building_condition', 'living_area', 'year_of_construction', 'energy_certificate', 'geolocation', 
                   'equipped_kitchen', 'bedroom_nr', 'swimming_pool', 'terrace', 'garden', 'plot_surface']

with open('property_data.csv', 'a', newline='') as file:
    csv.writer(file).writerow(field_names)

In [53]:
main_scrapper()

100%|██████████| 20/20 [00:04<00:00,  4.52it/s]


In [10]:
for l in test_urls[4:]:
    scrappy(l)

{'id': '20633288', 'price': '615000 - 765000', 'estate_type': 'av_10', 'distribution_type': '2', 'publication_id': 'IWB', 'name': 'classified', 'nb_bedrooms': '', 'nb_rooms': '', 'indoor_surface': '', 'zip_code': '1000', 'subtype': 'house_group', 'currency': 'eur', 'client_id': '2575488', 'client_type': 'pro', 'building_state': '', 'year_of_construction': '', 'energy_certificate': 'A', 'outdoor_terrace_exists': 'false', 'geolocation': '4.3458291,50.8435304', 'nb_picture': '11', 'rating': '', 'distribution_subtype': '2', 'energy': '', 'kitchen_type': '', 'land_surface': '', 'outdoor_surface': '', 'country': 'Belgium', 'province': 'Brussels', 'city': 'Bruxelles', 'parking': 'false', 'product_type': 'project'}
{'id': '20634289', 'price': '415000', 'estate_type': 'av_2', 'distribution_type': '2', 'publication_id': 'IWB', 'name': 'classified', 'nb_bedrooms': '1', 'nb_rooms': '', 'indoor_surface': '123', 'zip_code': '1000', 'subtype': 'apartment', 'currency': 'eur', 'client_id': '3444556', '

In [ ]:
real_estate = {'immo_id': None, 'zip_code': None, 'province':None, 'price': None, 'subtype_of_property': None, 'building_condition': None, 'living_area': None, 'year_of_construction': None, 
               'energy_certificate': None, 'geolocation': None, 'equipped_kitchen': 0, 'bedroom_nr': None, 'swimming_pool': 0, 'furnished': 0, 'terrace': 0, 'garden': 0, 'plot_surface': 0}

In [48]:
real_estate = {'immo_id': None, 'zip_code': None, 'province':None, 'price': None, 'subtype_of_property': None, 'building_condition': None, 'living_area': None, 'year_of_construction': None, 
               'energy_certificate': None, 'geolocation': None, 'equipped_kitchen': None, 'bedroom_nr': None, 'swimming_pool': 0, 'terrace': 0, 'garden': 0, 'plot_surface': 0}
lizd = []
for k in real_estate:
    lizd.append(k)

print(lizd)

['immo_id', 'zip_code', 'province', 'price', 'subtype_of_property', 'building_condition', 'living_area', 'year_of_construction', 'energy_certificate', 'geolocation', 'equipped_kitchen', 'bedroom_nr', 'swimming_pool', 'terrace', 'garden', 'plot_surface']


In [45]:
for u in test_urls:
    print(scrappy(u))

[['https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20529728'], ['https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20529731'], ['https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20529725'], ['https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20529726'], ['https://www.immoweb.be/en/classified/apartment/for-sale/bruxelles/1000/20529733']]
[['https://www.immoweb.be/en/classified/flat-studio/for-sale/bruxelles/1000/20632923'], ['https://www.immoweb.be/en/classified/flat-studio/for-sale/bruxelles/1000/20632952'], ['https://www.immoweb.be/en/classified/flat-studio/for-sale/bruxelles/1000/20632950'], ['https://www.immoweb.be/en/classified/flat-studio/for-sale/bruxelles/1000/20632921'], ['https://www.immoweb.be/en/classified/flat-studio/for-sale/bruxelles/1000/20632917'], ['https://www.immoweb.be/en/classified/flat-studio/for-sale/bruxelles/1000/20632951'], ['https://www.immoweb.be/en/classified/flat-studio/